# This Citi is Biking

## Research Question
How does Citi Bike usage differ between subscribers and casual riders, and how do riding patterns change by season and time of day?

## Dataset
Official Citi Bike System Data, January 2024–December 2025.

## Methodology
Parquet files were processed individually using Python and pandas. Data was aggregated to compare rider types, seasonal and monthly usage, time-of-day patterns, trip duration, and popular starting stations.

In [5]:
import pandas as pd
import os

In [3]:
import glob

files = glob.glob("../Data/processed/*.parquet")

print(files)

['../Data/processed/citibike_202409_2.parquet', '../Data/processed/citibike_202510_4.parquet', '../Data/processed/citibike_202405_1.parquet', '../Data/processed/citibike_202411_1.parquet', '../Data/processed/citibike_202406_5.parquet', '../Data/processed/citibike_202504_4.parquet', '../Data/processed/citibike_202402_3.parquet', '../Data/processed/citibike_202404_2.parquet', '../Data/processed/citibike_202506_3.parquet', '../Data/processed/citibike_202408_1.parquet', '../Data/processed/citibike_202509_4.parquet', '../Data/processed/citibike_202410_2.parquet', '../Data/processed/citibike_202512_3.parquet', '../Data/processed/citibike_202501_1.parquet', '../Data/processed/citibike_202404_3.parquet', '../Data/processed/citibike_202506_2.parquet', '../Data/processed/citibike_202410_3.parquet', '../Data/processed/citibike_202512_2.parquet', '../Data/processed/citibike_202509_5.parquet', '../Data/processed/citibike_202403_1.parquet', '../Data/processed/citibike_202510_5.parquet', '../Data/pro

In [6]:
all_data = []

for file in files:
    df = pd.read_parquet(file)
    all_data.append(df)

print("Number of Parquet files loaded:", len(all_data))

Number of Parquet files loaded: 103


In [4]:
df = pd.concat(all_data, ignore_index=True)

print(df.shape)

(90074311, 13)


In [8]:
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

Total rows: 1000000
Total columns: 13


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   ride_id             1000000 non-null  str    
 1   rideable_type       1000000 non-null  str    
 2   started_at          1000000 non-null  str    
 3   ended_at            1000000 non-null  str    
 4   start_station_name  999638 non-null   str    
 5   start_station_id    999638 non-null   str    
 6   end_station_name    1000000 non-null  str    
 7   end_station_id      999909 non-null   str    
 8   start_lat           999638 non-null   float64
 9   start_lng           999638 non-null   float64
 10  end_lat             999909 non-null   float64
 11  end_lng             999909 non-null   float64
 12  member_casual       1000000 non-null  str    
dtypes: float64(4), str(9)
memory usage: 228.2 MB


In [10]:
df.isna().sum()

ride_id                    0
rideable_type              0
started_at                 0
ended_at                   0
start_station_name     50106
start_station_id       50106
end_station_name      235900
end_station_id        252905
start_lat              50106
start_lng              50106
end_lat               252451
end_lng               252451
member_casual              0
dtype: int64

In [11]:
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])

df["ride_duration"] = (
    df["ended_at"] - df["started_at"]
).dt.total_seconds() / 60

df["month"] = df["started_at"].dt.month
df["year"] = df["started_at"].dt.year
df["hour"] = df["started_at"].dt.hour

df["season"] = df["month"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
})

In [13]:
print("Negative durations:", (df["ride_duration"] <= 0).sum())

print("Rides longer than 24 hours:", (df["ride_duration"] > 1440).sum())

Negative durations: 801
Rides longer than 24 hours: 20002


In [10]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

(1000000, 13)
['ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual']
ride_id                   str
rideable_type             str
started_at                str
ended_at                  str
start_station_name        str
start_station_id          str
end_station_name          str
end_station_id            str
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual             str
dtype: object


In [11]:
print("File list variable exists:", 'files' in globals())
print("DataFrame exists:", 'df' in globals())

File list variable exists: True
DataFrame exists: True


In [12]:
import pandas as pd
from pathlib import Path
from collections import Counter

In [13]:
output_folder = Path("../Output")
output_folder.mkdir(parents=True, exist_ok=True)

In [14]:
columns = [
    "started_at",
    "ended_at",
    "member_casual",
    "start_station_name"
]

In [15]:
rider_counts = Counter()
season_counts = Counter()
monthly_counts = Counter()
hourly_counts = Counter()
station_counts = Counter()

duration_totals = Counter()
duration_counts = Counter()

In [17]:
# Process each Parquet file individually
for i, file in enumerate(files, 1):

    part = pd.read_parquet(file, columns=columns)

    started = pd.to_datetime(part["started_at"], errors="coerce")
    ended = pd.to_datetime(part["ended_at"], errors="coerce")


In [29]:
# Keep only the project date range
valid = (
    started.notna()
        & (started >= "2024-01-01")
        & (started < "2026-01-01")
    )

started = started[valid]
ended = ended[valid]
riders = part.loc[valid, "member_casual"]
stations = part.loc[valid, "start_station_name"]

In [31]:
# 1. Rider comparison
rider_counts.update(riders.value_counts().to_dict())

    # 2. Seasonal ridership
seasons = started.dt.month.map({
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Spring", 4: "Spring", 5: "Spring",
        6: "Summer", 7: "Summer", 8: "Summer",
        9: "Fall", 10: "Fall", 11: "Fall"
    })
season_counts.update(seasons.value_counts().to_dict())


In [32]:
# 6. Trip duration
duration = (ended - started).dt.total_seconds() / 60

valid_duration = duration.notna() & (duration > 0)

duration_data = pd.DataFrame({
        "rider": riders[valid_duration],
        "duration": duration[valid_duration]
    })

duration_summary = duration_data.groupby("rider")["duration"].agg(
        ["sum", "count"]
    )

In [35]:
for rider, row in duration_summary.iterrows():
        duration_totals[rider] += row["sum"]
        duration_counts[rider] += row["count"]

del part, duration_data

if i % 10 == 0 or i == len(files):
        print(f"Processed {i} of {len(files)} files")

Processed 103 of 103 files


In [36]:
pd.DataFrame(
    rider_counts.items(),
    columns=["rider_type", "rides"]
).to_csv(output_folder / "rider_summary.csv", index=False)

pd.DataFrame(
    season_counts.items(),
    columns=["season", "rides"]
).to_csv(output_folder / "season_summary.csv", index=False)

pd.DataFrame(
    monthly_counts.items(),
    columns=["month", "rides"]
).sort_values("month").to_csv(
    output_folder / "monthly_summary.csv", index=False
)

pd.DataFrame(
    hourly_counts.items(),
    columns=["hour", "rides"]
).sort_values("hour").to_csv(
    output_folder / "hourly_summary.csv", index=False
)

pd.DataFrame([
    {
        "rider_type": rider,
        "average_duration_minutes": duration_totals[rider] / duration_counts[rider],
        "trip_count": duration_counts[rider]
    }
    for rider in duration_totals
]).to_csv(output_folder / "duration_summary.csv", index=False)

pd.DataFrame(
    station_counts.most_common(10),
    columns=["starting_station", "rides"]
).to_csv(output_folder / "station_summary.csv", index=False)

print("\nANALYSIS COMPLETE")
print("Summary files saved to:", output_folder.resolve())


ANALYSIS COMPLETE
Summary files saved to: /Users/fariharahman/Desktop/project.index/Citbike/Output


In [37]:
import pandas as pd
from pathlib import Path

output_folder = Path("../Output")

for filename in [
    "rider_summary.csv",
    "season_summary.csv",
    "monthly_summary.csv",
    "hourly_summary.csv",
    "duration_summary.csv",
    "station_summary.csv"
]:
    print("\n" + "=" * 40)
    print(filename)
    print("=" * 40)

    summary = pd.read_csv(output_folder / filename)
    print(summary.to_string(index=False))


rider_summary.csv
rider_type  rides
    member 772911
    casual 227089

season_summary.csv
season   rides
Summer 1000000

monthly_summary.csv
Empty DataFrame
Columns: [month, rides]
Index: []

hourly_summary.csv
Empty DataFrame
Columns: [hour, rides]
Index: []

duration_summary.csv
rider_type  average_duration_minutes  trip_count
    casual                 20.325603    227089.0
    member                 12.028799    772911.0

station_summary.csv
Empty DataFrame
Columns: [starting_station, rides]
Index: []


In [38]:
print("Number of files:", len(files))

print("First file:", files[0])
print("Last file:", files[-1])

print("Total rides in rider summary:",
      pd.read_csv("../Output/rider_summary.csv")["rides"].sum())

print("Monthly summary rows:",
      len(pd.read_csv("../Output/monthly_summary.csv")))

Number of files: 103
First file: ../Data/processed/citibike_202409_2.parquet
Last file: ../Data/processed/citibike_202508_2.parquet
Total rides in rider summary: 1000000
Monthly summary rows: 0
